In [1]:
import json
import os
from typing import List
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, messages_from_dict, messages_to_dict
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI

# =====================================================================

class JSONChatMessageHistory(BaseChatMessageHistory):
    """Кастомный класс истории сообщений, сохраняющий данные в JSON файл."""
    
    def __init__(self, session_id: str, file_directory: str = "chat_histories"):
        self.session_id = session_id
        self.file_directory = file_directory
        # Создаем папку для историй, если её нет
        os.makedirs(self.file_directory, exist_ok=True)
        self.file_path = os.path.join(self.file_directory, f"{session_id}.json")

    @property
    def messages(self) -> List[BaseMessage]:
        """Загружает сообщения из JSON-файла при каждом обращении."""
        if not os.path.exists(self.file_path):
            return []
        try:
            with open(self.file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                # Восстанавливаем объекты сообщений LangChain из словаря
                return messages_from_dict(data)
        except (json.JSONDecodeError, KeyError):
            return []

    def add_messages(self, messages: List[BaseMessage]) -> None:
        """Добавляет новые сообщения в JSON-файл."""
        # 1. Читаем текущую историю
        current_messages = self.messages
        # 2. Добавляем новые сообщения
        current_messages.extend(messages)
        # 3. Сериализуем объекты сообщений в стандартный формат словарей LangChain
        serializable_data = messages_to_dict(current_messages)
        # 4. Перезаписываем файл
        with open(self.file_path, "w", encoding="utf-8") as f:
            json.dump(serializable_data, f, ensure_ascii=False, indent=2)

    def clear(self) -> None:
        """Очищает историю (удаляет файл)."""
        if os.path.exists(self.file_path):
            os.remove(self.file_path)

# =====================================================================

llm = ChatOpenAI(
    model="maple",
    base_url="http://localhost:8080/v1", 
    api_key="not-needed",
    temperature=0.6,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты ИИ-ассистент. Отвечай лаконично."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_input}")
])

base_chain = prompt | llm

def get_json_history(session_id: str) -> JSONChatMessageHistory:
    """Функция-фабрика для RunnableWithMessageHistory."""
    return JSONChatMessageHistory(session_id=session_id)

# Оборачиваем цепочку для автоматической работы с нашей JSON-памятью
bot_chain = RunnableWithMessageHistory(
    base_chain,
    get_session_history=get_json_history,
    input_messages_key="user_input",
    history_messages_key="chat_history"
)
#
def chat(bot_chain):
    print("История сохраняется в папку 'chat_histories'.")
    print("Для выхода введите 'пока' 'выход', 'exit' или 'q'\n")

    SESSION_ID = input("Введите идентификатор\n>").strip()

    config = {"configurable": {"session_id": SESSION_ID}}

    print(f"Pei Kib Breat слушает...(Голос: {SESSION_ID})")

    while True:
        try:
            user_message = input("> ").strip()
            
            if user_message.lower() in ['пока', 'выход', 'exit', 'q']:
                print("Pei Kib Breat: Пока!")
                break
                
            if not user_message:
                continue
                
            # Отправка сообщения
            response = bot_chain.invoke(
                {"user_input": user_message}, 
                config=config
            )
            
            print(f"Pei Kib Breat: {response.content}\n")
            
        except KeyboardInterrupt:
            print("\nPei Kib Breat: Сессия завершена. Пока!")
            break

/home/a/anaconda3/envs/Statistic/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [3]:
chat(bot_chain) # Первый пользователь

История сохраняется в папку 'chat_histories'.
Для выхода введите 'пока' 'выход', 'exit' или 'q'



Введите идентификатор
> 1


Pei Kib Breat слушает...(Голос: 1)


>  Привет, меня зовут Семён


Pei Kib Breat: Привет, Семён! Чем могу помочь?



...

Pei Kib Breat: Привет, Семён! Очень интересно от биоинформатики — транскриптомный анализ, ML и статистика — это ключевые инструменты для современных исследований. Расскажи, чем занимаешься, могу подсказать или обсудить методы.



>  Я пишу на Python, R и bash


Pei Kib Breat: Отлично, Python, R и Bash — отличный набор для биоинформатики. Python популярен для ML и обработки данных, R для статистики и визуализации, Bash для автоматизации. 

С чем ты работаешь? Могу помочь с кодом, алгоритмами или анализом данных.



>  пока


Pei Kib Breat: Пока!


In [4]:
chat(bot_chain)# Второй пользователь

История сохраняется в папку 'chat_histories'.
Для выхода введите 'пока' 'выход', 'exit' или 'q'



Введите идентификатор
> 2


Pei Kib Breat слушает...(Голос: 2)


>  Привет, меня зовут Семен


Pei Kib Breat: Привет, Семён! Рад познакомиться. Чем могу помочь?



...

Pei Kib Breat: Привет, Семён! Рад познакомиться. Если будут вопросы по иммунологии или других темам — обращайся.



>  Я умею выделять клетки PBM из цельной крови, делать анализ лейкоцитов и работать на проточном цитометре


Pei Kib Breat: Отлично, это ключевая часть иммунологических исследований. Выборка PBMC (лимфоцитарных и моноцитарных клеток) и анализ лейкоцитов с последующей цитометрией — основа для иммунологических и иммунопротезных исследований. 

Работа с цитометром требует высокой точности и понимания структуры клетки. О чем именно вы хотите поговорить? В рамках своих экспериментов или в общении с коллегами?



>  пока


Pei Kib Breat: Пока!


In [5]:
chat(bot_chain) # Первый пользователь reuse

История сохраняется в папку 'chat_histories'.
Для выхода введите 'пока' 'выход', 'exit' или 'q'



Введите идентификатор
> 1


Pei Kib Breat слушает...(Голос: 1)


>  Привет, помнишь, как меня зовут?


Pei Kib Breat: Привет, Семён! Да, помню. Чем могу помочь?



...

Pei Kib Breat: Ты занимаешься биоинформатикой — анализом транскриптомов, машинным обучением и статистикой. Работаешь с данными, строишь модели, автоматизируешь процессы обработки геномных данных.



>  На каких языках я умею программировать?


Pei Kib Breat: Ты умеешь программировать на **Python, R, Bash** — это основные языки для твоей работы: Python для ML и анализа данных, R для статистики и визуализации, Bash для автоматизации пайплайнов. Есть и другие, но именно эти твои ключевые. Нужна помощь с кодом или объяснением?



>  пока


Pei Kib Breat: Пока!


In [6]:
chat(bot_chain) # Второй пользователь reuse

История сохраняется в папку 'chat_histories'.
Для выхода введите 'пока' 'выход', 'exit' или 'q'



Введите идентификатор
> 2


Pei Kib Breat слушает...(Голос: 2)


>  Привет, ты меня помнишь?


Pei Kib Breat: Запомню, Семён. Чем могу помочь?



...

Pei Kib Breat: Привет! Вот краткий перечень моих возможностей в контексте вашей сферы:

- **Изготовление PBMC** (лимфоцитарных и моноцитарных клеток) из целого крови  
- **Анализ лейкоцитов** (под разбор, подсчёт, морфология)  
- **Работа на иммуноцитрометре** (циточные стереометрии, анализ структуры/функции клеток)  
- **Иммунологические исследования** (например, макрофагов-активированных, T/NK-клеток, цитокинов)  
- **Обработка данных** (построение графиков, статистическая интерпретация)  

Если есть конкретная задача — напишите, разберём.



>  Нормально. Как посоветуешь выделять PBMC?


Pei Kib Breat: Изолируйте PBMC методом цитопероксизма с последующей резервуарной обработкой (например, сепаратом с использованием ферритина или полосаточного спирта). Ключевые моменты для воспроизводимости:

1. **Время** — в течение 1–2 часов после забора крови (лучше в холодном (4°C) виде).
2. **Температура** — холодная, но не морозная среда для разрушения лимфоцитарной сети.
3. **Расчёт** — PBMC ≈ (Hematocrit × 100) – (Hemoglobin × 0.5) — потом корректируйте по фактическому количеству клеток (обычно 1–2 × 10⁶/мл).
4. **Проверка** — по гемокриту (не должно быть лейкоцитарной фазы >20%) и осмотическому анализу.

Если нужно быстро получить большую численность — можно использовать commercial kit (например, Easy Sepharose), но учитывайте потери при резервуаре.



>  пока


Pei Kib Breat: Пока!


___